In [17]:
import os

In [18]:
%pwd

'p:\\OM\\Projects\\End-to-End-ML-'

In [19]:
os.chdir("../")

In [20]:
%pwd

'p:\\OM\\Projects'

In [35]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen = True) # frozen enures that once an instance of the following class is created the class attributes can't be modified.
class DataIngestionConfig:
    root_dir : Path
    src_url : str
    local_data_file : Path
    unzip_dir : Path

In [ ]:
from src.wine.constants import *
from src.wine.utils.common import read_yaml, create_directories

In [36]:
class ConfigurationManager:
    def __init__(
        self,
        config_file_path = CONFIG_FILE_PATH,
        params_file_path = PARAMS_FILE_PATH,
        schema_file_path = SCHEMA_FILE_PATH):

        self.config = read_yaml(config_file_path)
        self.params = read_yaml(params_file_path)
        self.schema = read_yaml(schema_file_path)

        create_directories([self.config.artifacts_root])

    def get_data_ingestion_config(self) -> DataIngestionConfig:
        config = self.config.data_ingestion

        create_directories([config.root_dir])

        data_ingestion_config = DataIngestionConfig(
            root_dir=config.root_dir,
            src_url=config.src_url,
            local_data_file=config.local_data_file,
            unzip_dir=config.unzip_dir 
        )

        return data_ingestion_config

In [26]:
import os
import urllib.request as request
import zipfile
from src.wine.logging import logger
from src.wine.utils.common import get_size

In [ ]:
class DataIngestion:
    def __init__(self, config: DataIngestionConfig):
        self.config = config


    
    def download_file(self):
        if not os.path.exists(self.config.local_data_file):
            filename, headers = request.urlretrieve(
                url = self.config.src_url,
                filename = self.config.local_data_file
            )
            logger.info(f"{filename} download! with following info: \n{headers}")
        else:
            logger.info(f"File already exists of size: {get_size(Path(self.config.local_data_file))}")



    
    def extract_zip_file(self):
        """
        zip_file_path: str
        Extracts the zip file into the data directory
        Function returns None
        """
        unzip_path = self.config.unzip_dir
        os.makedirs(unzip_path, exist_ok=True)
        with zipfile.ZipFile(self.config.local_data_file, 'r') as zip_ref:
            zip_ref.extractall(unzip_path)

In [40]:
try : 
    config = ConfigurationManager()     # object creation
    data_ingestion_config = config.get_data_ingestion_config()      # stored output of config object's method
    data_ingestion = DataIngestion(config = data_ingestion_config)     # object creation
    data_ingestion.download_file()
    data_ingestion.extract_zip_file()

except Exception as e : 
    raise e 

[ 2026-06-21 01:07:16,412 : INFO : common : yaml file: config\config.yaml loaded successfully ]
[ 2026-06-21 01:07:16,414 : INFO : common : yaml file: params.yaml loaded successfully ]
[ 2026-06-21 01:07:16,417 : INFO : common : yaml file: schema.yaml loaded successfully ]
[ 2026-06-21 01:07:16,418 : INFO : common : created directory at: artifacts ]
[ 2026-06-21 01:07:16,419 : INFO : common : created directory at: artifacts/data_ingestion ]
[ 2026-06-21 01:07:17,610 : INFO : 2747247313 : artifacts/data_ingestion/wine_data.zip download! with following info: 
Connection: close
Content-Length: 21315
Cache-Control: max-age=300
Content-Security-Policy: default-src 'none'; style-src 'unsafe-inline'; sandbox
Content-Type: application/zip
ETag: "30d4090c59d08cd0729bba3d7162b2300c3f6c9320d09134ebe40eac45895761"
Strict-Transport-Security: max-age=31536000
X-Content-Type-Options: nosniff
X-Frame-Options: deny
X-XSS-Protection: 1; mode=block
X-GitHub-Request-Id: D5FC:359F8E:17799D:3D0828:6A36EBED
